In [4]:
from scipy.io import loadmat

data = loadmat("../p2data/dataset.mat")

print(data.keys())
print(data["EEGsample"].shape)
print(data["substate"].shape)
print(data["subindex"].shape)

dict_keys(['__header__', '__version__', '__globals__', 'EEGsample', 'subindex', 'substate'])
(2022, 30, 384)
(2022, 1)
(2022, 1)


In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import welch

bands = {
    "theta": (4, 7),
    "alpha": (8, 12),
    "beta": (13, 30),
}

def extract_sample_features(sample, sfreq=128):
    features = {}

    for ch in range(sample.shape[0]):
        signal = sample[ch]

        freqs, psd = welch(signal, fs=sfreq, nperseg=256)

        for band_name, (fmin, fmax) in bands.items():
            idx = (freqs >= fmin) & (freqs <= fmax)
            band_power = psd[idx].mean()

            features[f"{band_name}_ch{ch}"] = band_power

    return features

rows = []

EEG = data["EEGsample"]
labels = data["substate"].flatten()
subjects = data["subindex"].flatten()

for i in range(EEG.shape[0]):
    sample = EEG[i]

    features = extract_sample_features(sample)

    row = {
        "subject": int(subjects[i]),
        "label": int(labels[i]),
        **features
    }

    rows.append(row)

df_drowsy = pd.DataFrame(rows)

print(df_drowsy.shape)
df_drowsy.head()

(2022, 92)


,subject,label,theta_ch0,alpha_ch0,beta_ch0,theta_ch1,alpha_ch1,beta_ch1,theta_ch2,alpha_ch2,...,beta_ch26,theta_ch27,alpha_ch27,beta_ch27,theta_ch28,alpha_ch28,beta_ch28,theta_ch29,alpha_ch29,beta_ch29
0,1,0,2.333423,1.313866,0.255312,2.389377,1.072117,0.262868,1.604459,0.952285,...,0.257146,0.918626,0.855799,0.478499,1.017187,1.054977,0.411086,1.027991,1.256180,0.329739
1,1,0,1.062462,0.835138,0.312658,0.809725,0.769227,0.322722,1.155921,0.728720,...,0.251492,0.425243,0.432578,0.394842,0.499038,0.497478,0.394829,0.526728,0.362649,0.334843
2,1,0,2.765192,1.292331,0.424175,3.283794,1.024831,0.415939,4.247937,1.368840,...,0.353195,0.486668,1.929664,0.524787,0.614498,1.964493,0.476765,1.060039,2.825614,0.453383
3,1,0,0.543084,0.804349,0.455062,0.444233,0.872507,0.477724,0.432821,0.656188,...,0.447785,0.585675,0.825440,0.727546,0.490133,0.554237,0.779283,0.548036,0.575963,0.724429
4,1,0,1.222005,1.394455,0.436912,1.319642,1.246246,0.442536,0.949002,1.057056,...,0.297115,1.017788,1.059616,0.539397,0.767081,1.004317,0.460188,0.622509,1.076213,0.360496


In [ ]:
from sklearn.model_selection import LeaveOneGroupOut, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
#Logistic Regression
feature_cols = [c for c in df_drowsy.columns if c not in ["subject", "label"]]

X = df_drowsy[feature_cols].values
y = df_drowsy["label"].values
groups = df_drowsy["subject"].values

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

logo = LeaveOneGroupOut()

scores = cross_val_score(
    model,
    X,
    y,
    cv=logo.split(X, y, groups=groups)
)

print("LOSO accuracies:", scores)
print("Mean LOSO accuracy:", scores.mean())

LOSO accuracies: [0.80851064 0.42424242 0.70666667 0.72297297 0.77232143 0.72289157
 0.60784314 0.64015152 0.86305732 0.75       0.52212389]
Mean LOSO accuracy: 0.6855255970971742


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import LeaveOneGroupOut, cross_val_score

#Random Forest
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

logo = LeaveOneGroupOut()

scores_rf = cross_val_score(
    model,
    X,
    y,
    cv=logo.split(X, y, groups=groups)
)

print("RF LOSO accuracies:", scores_rf)
print("RF Mean LOSO accuracy:", scores_rf.mean())

RF LOSO accuracies: [0.79787234 0.51515152 0.68       0.60810811 0.73660714 0.79518072
 0.6372549  0.65530303 0.88535032 0.82407407 0.60619469]
RF Mean LOSO accuracy: 0.7037360767735071


In [8]:
from sklearn.svm import SVC

model = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf")
)

scores_svm = cross_val_score(
    model,
    X,
    y,
    cv=logo.split(X, y, groups=groups)
)

print("SVM Mean LOSO accuracy:", scores_svm.mean())

SVM Mean LOSO accuracy: 0.7064103433552714


In [9]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=feature_cols)
print(importances.sort_values(ascending=False).head(20))

alpha_ch28    0.034545
theta_ch24    0.025756
alpha_ch27    0.023298
theta_ch8     0.021642
theta_ch7     0.021419
theta_ch17    0.021229
alpha_ch7     0.020855
theta_ch23    0.020344
alpha_ch29    0.019788
alpha_ch14    0.019355
theta_ch26    0.019059
theta_ch28    0.018418
theta_ch13    0.018003
alpha_ch24    0.017135
beta_ch7      0.017072
beta_ch2      0.015806
theta_ch27    0.015299
theta_ch14    0.015207
theta_ch9     0.014875
beta_ch21     0.014413
dtype: float64


In [16]:
top_features = importances.sort_values(ascending=False).head(10).index.tolist()

X_top = df_drowsy[top_features].values

scores_top = cross_val_score(
    make_pipeline(StandardScaler(), SVC(kernel="rbf")),
    X_top,
    y,
    cv=logo.split(X_top, y, groups=groups)
)

print("Top-feature SVM LOSO:", scores_top.mean())

Top-feature SVM LOSO: 0.7213226967816626
